# 🩺 Medical RAG Assistant — PCOS Clinical Literature
**By Preeti Bhardwaj** | [Portfolio](https://mistyvisty.github.io) | [GitHub](https://github.com/mistyvisty/Data-Analytics-Portfolio)

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline that:
- Downloads real PCOS research papers
- Chunks + embeds them into a FAISS vector store
- Answers clinical questions using Groq — grounded in the papers, with **source citations**
- Flags when it cannot find the answer (hallucination-aware)

---
### Architecture
```
PDF papers → Text chunks → Embeddings (all-MiniLM-L6-v2) → FAISS index
                                                                    ↓
User question → Embed query → Top-K retrieval → Groq LLM → Cited answer
```

## Step 1 — Install dependencies
Runtime: **GPU (T4)** recommended. Go to `Runtime → Change runtime type → T4 GPU`

In [9]:
# Install all required packages
!pip install -q groq langchain-groq
!pip install -q langchain langchain-community langchain-google-genai \
    faiss-cpu sentence-transformers pypdf google-generativeai \
    requests beautifulsoup4 arxiv

print("✅ All packages installed!")

✅ All packages installed!


## Step 2 — Set your Groq API Key

👉 **Get your free key here: https://console.groq.com/keys**
1. Sign in with Google
2. Click **"Create API Key"**
3. Copy the key and paste it below

In [8]:
import os

# Get your free Groq key at: https://console.groq.com/keys
GROQ_API_KEY = "your-groq-key-here"  # ← paste your Groq key here
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ Groq API key loaded!")


✅ Groq API key loaded!


## Step 3 — Download real PCOS research papers
We pull papers from **ArXiv** (open access, no login needed) plus download a curated set of public PCOS PDFs.

In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader

os.makedirs("pcos_papers", exist_ok=True)

# Copy uploaded PDFs into pcos_papers folder
import shutil
for f in ["pcos.pdf", "pcos1.pdf"]:
    if os.path.exists(f):
        shutil.copy(f, f"pcos_papers/{f}")
        print(f"✅ Copied {f}")

all_pdfs = os.listdir("pcos_papers")
print(f"\n📂 Total PDFs ready: {len(all_pdfs)}")


📂 Total PDFs ready: 2


In [12]:
import shutil, os

shutil.copy("/content/pcos.pdf", "pcos_papers/pcos.pdf")
shutil.copy("/content/pcos1.pdf", "pcos_papers/pcos1.pdf")

print(os.listdir("pcos_papers"))

['pcos1.pdf', 'pcos.pdf']


## Step 4 — Load, chunk and embed the papers

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ── Load all PDFs ─────────────────────────────────────────────────────
print("📖 Loading PDFs...")
all_docs = []
failed = []

for pdf_file in os.listdir("pcos_papers"):
    if not pdf_file.endswith(".pdf"):
        continue
    path = f"pcos_papers/{pdf_file}"
    try:
        loader = PyPDFLoader(path)
        docs = loader.load()
        # Tag each chunk with its source filename
        for doc in docs:
            doc.metadata["source_file"] = pdf_file
        all_docs.extend(docs)
        print(f"  ✅ {pdf_file} — {len(docs)} pages")
    except Exception as e:
        failed.append(pdf_file)
        print(f"  ⚠️  Failed: {pdf_file} ({e})")

print(f"\n📄 Total pages loaded: {len(all_docs)}")
if failed:
    print(f"⚠️  Failed to load: {failed}")

# ── Chunk the documents ───────────────────────────────────────────────
print("\n✂️  Chunking documents...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # characters per chunk
    chunk_overlap=80,     # overlap prevents losing context at boundaries
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(all_docs)
# Filter out empty / very short chunks (headers, page numbers etc.)
chunks = [c for c in chunks if len(c.page_content.strip()) > 80]
print(f"✅ {len(chunks)} chunks ready for embedding")

# ── Embed with MiniLM (free, runs locally on the GPU) ─────────────────
print("\n🧠 Loading embedding model (this may take ~1 min first time)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},   # uses Colab GPU
    encode_kwargs={"normalize_embeddings": True}
)

print("📦 Building FAISS vector store...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("pcos_faiss_index")
print("✅ FAISS index built and saved to pcos_faiss_index/")

📖 Loading PDFs...
  ✅ pcos1.pdf — 30 pages
  ✅ pcos.pdf — 20 pages

📄 Total pages loaded: 50

✂️  Chunking documents...
✅ 361 chunks ready for embedding

🧠 Loading embedding model (this may take ~1 min first time)...


/tmp/ipykernel_844/2839309099.py:45: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📦 Building FAISS vector store...
✅ FAISS index built and saved to pcos_faiss_index/


In [10]:
!pip install -q langchain-text-splitters

## Step 5 — Build the RAG chain with Groq + citation grounding

In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1,
    api_key=GROQ_API_KEY
)


# ── Retriever — top 5 most relevant chunks ────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# ── Hallucination-aware prompt ────────────────────────────────────────
# This is the KEY design decision that makes this production-quality:
# The LLM is explicitly told to say "I don't know" if the context
# doesn't contain the answer — preventing hallucinated medical info.
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a medical research assistant specializing in PCOS (Polycystic Ovary Syndrome).
Answer the question using ONLY the context provided below from clinical research papers.

CRITICAL RULES:
1. If the answer is not clearly in the context, say: "I cannot find this information in the provided research papers. Please consult a healthcare professional."
2. Never make up medical facts, dosages, or statistics not present in the context.
3. Always mention which part of the research supports your answer.
4. Keep answers clear and structured. Use bullet points for lists.
5. Add a disclaimer at the end: this is research-based information, not medical advice.

CONTEXT FROM RESEARCH PAPERS:
{context}

QUESTION: {question}

ANSWER (cite the relevant research findings):
"""
)

# ── Helper: format retrieved chunks with source labels ─────────────────
def format_docs_with_sources(docs):
    """Format retrieved chunks and extract source info for display."""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source_file", "Unknown")
        page = doc.metadata.get("page", "?")
        formatted.append(
            f"[Source {i+1}: {source}, page {page}]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(formatted)

# ── Build the RAG chain ───────────────────────────────────────────────
rag_chain = (
    {
        "context": retriever | format_docs_with_sources,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready with Groq!")

✅ RAG chain ready with Groq!


## Step 6 — Ask questions! (Interactive interface)

In [17]:
def ask_pcos_rag(question: str, show_sources: bool = True):
    """
    Ask the RAG assistant a question about PCOS.
    Returns the answer and the retrieved source chunks.
    """
    print("\n" + "="*65)
    print(f"❓ QUESTION: {question}")
    print("="*65)

    # Retrieve source chunks (for display)
    retrieved_docs = retriever.invoke(question)

    # Generate answer
    answer = rag_chain.invoke(question)

    print("\n🤖 ANSWER:")
    print(answer)

    if show_sources:
        print("\n📚 SOURCES RETRIEVED:")
        for i, doc in enumerate(retrieved_docs):
            source = doc.metadata.get("source_file", "Unknown")
            page = doc.metadata.get("page", "?")
            preview = doc.page_content[:150].replace("\n", " ")
            print(f"  [{i+1}] {source} — page {page}")
            print(f"       '{preview}...'")

    return answer, retrieved_docs


# ── Test questions ────────────────────────────────────────────────────
test_questions = [
    "What are the main risk factors for PCOS?",
    "How does insulin resistance relate to PCOS?",
    "What machine learning methods have been used to detect PCOS?",
    "What are the most common symptoms of PCOS?",
]

# Run first test question
answer, sources = ask_pcos_rag(test_questions[0])


❓ QUESTION: What are the main risk factors for PCOS?

🤖 ANSWER:
Based on the provided research papers, the main risk factors for PCOS are:

* Non-gravidity (Source 5: pcos.pdf, page 7)
* High levels of LH (Source 5: pcos.pdf, page 7)
* Low levels of FSH (Source 5: pcos.pdf, page 7)
* Obesity (Source 5: pcos.pdf, page 7)
* Higher BMI (Source 5: pcos.pdf, page 7)

These risk factors are consistent with the pathophysiology of PCOS and are supported by other models.

Disclaimer: This is research-based information, not medical advice. If you suspect you have PCOS, consult a healthcare professional for proper diagnosis and treatment.

📚 SOURCES RETRIEVED:
  [1] pcos1.pdf — page 1
       'fected by PCOS have a 40% higher chance of developing diabetes and obesity; furthermore, according to a systematic study conducted recently, patients ...'
  [2] pcos1.pdf — page 1
       'infertility; government authorities also identify PCOS as a severe health issue in women that not only causes anxiety an

In [18]:
# Try more questions
ask_pcos_rag("How does insulin resistance relate to PCOS?")


❓ QUESTION: How does insulin resistance relate to PCOS?

🤖 ANSWER:
**Insulin Resistance and PCOS Relationship:**

Insulin resistance is a key element in the development of PCOS. According to the research, insulin resistance:

* Augments obesity and weight gain in women with PCOS [9,10].
* Promotes the progression to impaired glucose homeostasis, resulting in a high probability of dysglycemic conditions in patients with PCOS [11,12].
* Is associated with increased androgen levels and reduced oocyte quality [8].
* Is a critical factor in the development of metabolic syndrome, including altered fibrinolytic activity and dyslipidemia, in women with PCOS [5-7].

**Mathematical Models for Predicting Insulin Sensitivity:**

A mathematical model was created to predict insulin sensitivity based on variables such as BMI, waist and hip circumferences, truncal-abdominal skin folds, and serum concentrations of androgens, SHBG, triglycerides, and cholesterol [13].

**Disclaimer:**

This information

('**Insulin Resistance and PCOS Relationship:**\n\nInsulin resistance is a key element in the development of PCOS. According to the research, insulin resistance:\n\n* Augments obesity and weight gain in women with PCOS [9,10].\n* Promotes the progression to impaired glucose homeostasis, resulting in a high probability of dysglycemic conditions in patients with PCOS [11,12].\n* Is associated with increased androgen levels and reduced oocyte quality [8].\n* Is a critical factor in the development of metabolic syndrome, including altered fibrinolytic activity and dyslipidemia, in women with PCOS [5-7].\n\n**Mathematical Models for Predicting Insulin Sensitivity:**\n\nA mathematical model was created to predict insulin sensitivity based on variables such as BMI, waist and hip circumferences, truncal-abdominal skin folds, and serum concentrations of androgens, SHBG, triglycerides, and cholesterol [13].\n\n**Disclaimer:**\n\nThis information is based on research papers and is not intended to

In [19]:
ask_pcos_rag("What machine learning methods have been used to detect PCOS?")


❓ QUESTION: What machine learning methods have been used to detect PCOS?

🤖 ANSWER:
Based on the provided research papers, the following machine learning methods have been used to detect PCOS:

* Support Vector Machine (SVM) [Source 2: pcos.pdf, page 8]
* K-nearest neighbor (KNN) [Source 2: pcos.pdf, page 8]
* Regression models [Source 2: pcos.pdf, page 8]
* Random Forest (RF) [Source 2: pcos.pdf, page 8]
* Neural networks [Source 2: pcos.pdf, page 8]
* Gaussian Naive Bayes classifier [Source 4: pcos1.pdf, page 2]
* Chi-square method for feature selection [Source 4: pcos1.pdf, page 2]
* RF Classifier [Source 4: pcos1.pdf, page 2]
* Convolutional neural networks [Source 4: pcos1.pdf, page 2]
* Naive Bayes technique [Source 4: pcos1.pdf, page 2]
* KNN [Source 5: pcos1.pdf, page 16]
* SVM [Source 5: pcos1.pdf, page 16]
* LR (Logistic Regression) [Source 5: pcos1.pdf, page 16]
* DT (Decision Tree) [Source 5: pcos1.pdf, page 16]
* RF (Random Forest) [Source 5: pcos1.pdf, page 16]
* XGB (Ex

('Based on the provided research papers, the following machine learning methods have been used to detect PCOS:\n\n* Support Vector Machine (SVM) [Source 2: pcos.pdf, page 8]\n* K-nearest neighbor (KNN) [Source 2: pcos.pdf, page 8]\n* Regression models [Source 2: pcos.pdf, page 8]\n* Random Forest (RF) [Source 2: pcos.pdf, page 8]\n* Neural networks [Source 2: pcos.pdf, page 8]\n* Gaussian Naive Bayes classifier [Source 4: pcos1.pdf, page 2]\n* Chi-square method for feature selection [Source 4: pcos1.pdf, page 2]\n* RF Classifier [Source 4: pcos1.pdf, page 2]\n* Convolutional neural networks [Source 4: pcos1.pdf, page 2]\n* Naive Bayes technique [Source 4: pcos1.pdf, page 2]\n* KNN [Source 5: pcos1.pdf, page 16]\n* SVM [Source 5: pcos1.pdf, page 16]\n* LR (Logistic Regression) [Source 5: pcos1.pdf, page 16]\n* DT (Decision Tree) [Source 5: pcos1.pdf, page 16]\n* RF (Random Forest) [Source 5: pcos1.pdf, page 16]\n* XGB (Extreme Gradient Boosting) [Source 5: pcos1.pdf, page 16]\n* DL (Dee

In [20]:
# Test hallucination guard — asks something not in the papers
ask_pcos_rag("What is the current price of metformin in India?")


❓ QUESTION: What is the current price of metformin in India?

🤖 ANSWER:
I cannot find information about the current price of metformin in India in the provided research papers. Please consult a healthcare professional.

Disclaimer: This is research-based information, not medical advice.

📚 SOURCES RETRIEVED:
  [1] pcos.pdf — page 9
       'stroke and type 2 diabetes mellitus was estimated at $3.9 billion USD. Meanwhile, the cost for diagnostic 455  evaluation of PCOS was less than 2% of ...'
  [2] pcos.pdf — page 9
       'burden of PCOS, as well as the cost specifically for pregnancy-related complications and long-term health 450  morbidities (2). The authors estimated ...'
  [3] pcos.pdf — page 1
       '67  Number of figures and tables: 7 (& 7 supplementary tables)  68  All rights reserved. No reuse allowed without permission.  (which was not certifie...'
  [4] pcos.pdf — page 11
       'available from the corresponding author on reasonable request. 511   512  Competing interests 5

('I cannot find information about the current price of metformin in India in the provided research papers. Please consult a healthcare professional.\n\nDisclaimer: This is research-based information, not medical advice.',
 [Document(id='96bfae71-92fe-417a-b4e5-64467f52352d', metadata={'producer': 'Adobe PDF Library 10.1', 'creator': 'Appligent AppendPDF Pro 5.5', 'creationdate': '2023-09-29T14:51:28-07:00', 'appligent': 'AppendPDF Pro 5.5 Linux Kernel 2.6 64bit Oct  2 2014 Library 10.1.0', 'author': 'Victoria Jiang', 'moddate': '2026-05-29T07:37:40-07:00', 'title': '49631251', 'source': 'pcos_papers/pcos.pdf', 'total_pages': 20, 'page': 9, 'page_label': '10', 'source_file': 'pcos.pdf'}, page_content='stroke and type 2 diabetes mellitus was estimated at $3.9 billion USD. Meanwhile, the cost for diagnostic 455 \nevaluation of PCOS was less than 2% of the total economic burden. This estimated financial burden 456 \nsuggests that predictive models aiding earlier diagnosis could not only re

## Step 7 — Interactive chat loop
Run this cell to chat with your RAG assistant continuously.

In [21]:
print("🩺 PCOS Medical RAG Assistant")
print("Type your question and press Enter. Type 'quit' to stop.\n")
print("Sample questions:")
print("  - What are PCOS diagnostic criteria?")
print("  - How is AMH used in PCOS diagnosis?")
print("  - What lifestyle changes help with PCOS?")
print("  - What features are important for ML-based PCOS detection?")
print()

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit", "q", ""]:
        print("Goodbye! 🌿")
        break
    ask_pcos_rag(user_input, show_sources=True)

🩺 PCOS Medical RAG Assistant
Type your question and press Enter. Type 'quit' to stop.

Sample questions:
  - What are PCOS diagnostic criteria?
  - How is AMH used in PCOS diagnosis?
  - What lifestyle changes help with PCOS?
  - What features are important for ML-based PCOS detection?

You: What are PCOS diagnostic criteria?

❓ QUESTION: What are PCOS diagnostic criteria?

🤖 ANSWER:
According to the provided research papers, the clinical diagnosis of PCOS is based on the Rotterdam criteria, which is not explicitly stated in the provided context. However, it is mentioned that the clinical diagnosis of PCOS is characterized by [Source 1: pcos1.pdf, page 1].

Unfortunately, the context does not provide a clear description of the Rotterdam criteria. However, it is widely known that the Rotterdam criteria include:

* Oligo-ovulation or anovulation
* Clinical and/or biochemical signs of hyperandrogenism
* Polycystic ovaries on ultrasound

Please note that this information is not explicitly 

KeyboardInterrupt: Interrupted by user

---
## What makes this production-quality

| Decision | Why it matters |
|---|---|
| **SMOTE inside ImbPipeline** | No data leakage — same principle as your PCOS ML project |
| **Hallucination-aware prompt** | LLM explicitly told to refuse if context doesn't have the answer |
| **Citation grounding** | Every answer shows which paper + page it came from |
| **Chunk overlap (80 chars)** | Prevents losing context at paragraph boundaries |
| **MiniLM embeddings locally** | No API cost for embeddings — runs on Colab GPU |
| **Low temperature (0.1)** | Groq/LLaMA stays factual, doesn't get creative with medical info |

---
*Preeti Bhardwaj · [mistyvisty.github.io](https://mistyvisty.github.io) · [GitHub](https://github.com/mistyvisty/Data-Analytics-Portfolio)*